# Learn legal reasoning by writing code
**A hands-on tutorial with PYTHEN 2 (v0.2.0a3)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nguyenthanhasia/pythen/blob/main/tutorials/legal_reasoning/legal_reasoning.ipynb)

A customer requests a refund. Is the claim established? What evidence is missing? What additional evidence could reverse the answer? You will learn how to formalize these questions, evaluate them and build a small decision-support tool.

**Prerequisites:** basic Python, no formal logic or law background required. Work through the cells in order, predict each result before running it, and try the exercises. The rules are hypothetical and are not legal advice or the law of any particular jurisdiction. This notebook pins an alpha research release for reproducibility.

In [ ]:
%pip install -q "pythen==0.2.0a3"

In [ ]:
from importlib.metadata import version
from pythen import Reasoner, Intervention
from IPython.display import display, Markdown
from itertools import product
assert version("pythen") == "0.2.0a3"
print("PYTHEN", version("pythen"))

## 1. From a legal sentence to propositions
Start with a simplified policy: a buyer qualifies for a refund if a product is defective **and** the purchase is established. Before coding, separate the conclusion from its conditions.

| Type | Proposition | Example evidence |
|---|---|---|
| Base fact | `product_defective` | Inspection report |
| Base fact | `purchase_established` | Receipt or transaction record |
| Derived conclusion | `refund_entitlement` | Application of the rule |

A proposition is a statement evaluated within a model. Evidence is what supports establishing a fact; do not confuse a document with the proposition it supports.

In [ ]:
TARGET = "refund_entitlement"
basic_rules = [{
    "p": TARGET, "op": "ALL",
    "conditions": ["product_defective", "purchase_established"],
    "exceptions": []
}]

def evaluate(rules, facts, target=TARGET):
    return Reasoner(rules, facts=facts).evaluate(target)

assert evaluate(basic_rules, ["product_defective", "purchase_established"]) is True
assert evaluate(basic_rules, ["product_defective"]) is False
print(evaluate(basic_rules, ["product_defective", "purchase_established"]))

## 2. AND and OR
`ALL` means every listed condition must hold. `ANY` means at least one must hold. Suppose either a defect or a product misdescription is a sufficient return ground. Predict each row before running the table below.

In [ ]:
any_rules = [{
    "p": "valid_return_ground", "op": "ANY",
    "conditions": ["product_defective", "product_misdescribed"],
    "exceptions": []
}]
lines = ["| A | B | ALL | ANY |", "|---|---|---|---|"]
for a, b in product([False, True], repeat=2):
    all_facts = [name for name, value in zip(
        ["product_defective", "purchase_established"], [a, b]) if value]
    any_facts = [name for name, value in zip(
        ["product_defective", "product_misdescribed"], [a, b]) if value]
    x = evaluate(basic_rules, all_facts)
    y = evaluate(any_rules, any_facts, "valid_return_ground")
    assert x == (a and b) and y == (a or b)
    lines.append(f"| {a} | {b} | {x} | {y} |")
display(Markdown("\n".join(lines)))

## 3. Exceptions and defeasible reasoning
Amend the first rule: the buyer qualifies unless the buyer caused the defect. Adding a fact may defeat a previously established conclusion. This is non-monotonic reasoning.

In [ ]:
exception_rules = [{
    "p": TARGET, "op": "ALL",
    "conditions": ["product_defective", "purchase_established"],
    "exceptions": ["buyer_caused_defect"]
}]
original = Reasoner(exception_rules, facts=["product_defective", "purchase_established"])
scenario = original.intervene(Intervention(add=["buyer_caused_defect"]))
assert original.evaluate(TARGET) is True
assert scenario.evaluate(TARGET) is False
print("Original:", original.evaluate(TARGET))
print("After exception:", scenario.evaluate(TARGET))

**Important:** PYTHEN uses a closed-world Boolean model: an absent leaf fact evaluates to false. A missing fact is not proof that it is false in the real world. Here, an exception must be established to block the rule. Whether that matches the applicable burden of proof must be assessed separately. A false result means the model does not establish the claim, not necessarily that the person has no legal right.

## 4. Build a reusable reasoning graph
Refine the hypothetical policy: a defect or a misdescription is a valid return ground. A documented exception defeats **only the defect ground** when the buyer caused the defect and the damage occurred after delivery. A separate misdescription ground remains available. Purchase must be established in either case.

This is a deliberately simplified policy, not actual law. Notice how exception scope affects the graph:

```text
refund_entitlement (ALL)
  |-- purchase_established
  `-- valid_return_ground (ANY)
        |-- product_misdescribed
        `-- defect_ground (ALL)
              |-- product_defective
              `-- EXCEPTION: damage_exception (ALL)
                    |-- buyer_caused_defect
                    `-- damage_after_delivery
```

In [ ]:
rules = [
    {"p": TARGET, "op": "ALL",
     "conditions": ["purchase_established", "valid_return_ground"], "exceptions": []},
    {"p": "valid_return_ground", "op": "ANY",
     "conditions": ["defect_ground", "product_misdescribed"], "exceptions": []},
    {"p": "defect_ground", "op": "ALL",
     "conditions": ["product_defective"], "exceptions": ["damage_exception"]},
    {"p": "damage_exception", "op": "ALL",
     "conditions": ["buyer_caused_defect", "damage_after_delivery"], "exceptions": []},
]
BASE = ["product_defective", "purchase_established"]
EXCEPTION = ["buyer_caused_defect", "damage_after_delivery"]
FACT_UNIVERSE = ["product_defective", "purchase_established", "product_misdescribed", *EXCEPTION]

## 5. Read a verdict as a legal argument
Predict five outcomes. Alice and Bob have different grounds. Charlie is missing a condition. David has an established exception. Emma has an independent misdescription ground despite the exception.

In [ ]:
cases = {
    "Alice": (BASE, True),
    "Bob": (["product_misdescribed", "purchase_established"], True),
    "Charlie": (["product_defective"], False),
    "David": (BASE + EXCEPTION, False),
    "Emma": (BASE + EXCEPTION + ["product_misdescribed"], True),
}
lines = ["| Case | Purchase | Defect ground | Misdescription | Exception | Entitlement |",
         "|---|---|---|---|---|---|"]
for name, (facts, expected) in cases.items():
    case = Reasoner(rules, facts=facts)
    verdict = case.evaluate(TARGET)
    assert verdict is expected, name
    values = [case.evaluate(p) for p in ["purchase_established", "defect_ground",
                                         "product_misdescribed", "damage_exception", TARGET]]
    lines.append("| " + " | ".join([name] + ["Established" if x else "Not established" for x in values]) + " |")
display(Markdown("\n".join(lines)))

## 6. Ask what would change the conclusion
Counterfactual reasoning asks what would happen if facts were different. Here we ask a narrower, addition-only question: what is the smallest set of absent, reachable leaf facts that changes the verdict?

`radius` counts additions; `witness` supplies one minimum set. Predict the answer for Alice. Can one extra fact block the defect ground?

In [ ]:
alice = Reasoner(rules, facts=BASE)
result = alice.challenge(TARGET)
assert result.status == "flipped" and result.radius == 2
assert set(result.witness) == set(EXCEPTION)
print("Status:", result.status)
print("Radius:", result.radius)
print("Witness:", result.witness)

In [ ]:
check = alice.verify(result)
assert check.valid is True and check.optimal is True
assert alice.intervene(Intervention(add=result.witness)).evaluate(TARGET) is False
assert alice.evaluate(TARGET) is True
print("Witness valid:", check.valid, "Minimum verified:", check.optimal)

Validity checks whether the intervention flips the result. Optimality checks whether a smaller permitted intervention exists. Both are relative to the encoded rules, original facts and candidate set. They do not establish legal correctness, evidence credibility, or the probability of winning a case. The verification uses the same Boolean semantics, not an independent legal interpretation.

## 7. Constrain the search
A lawyer might be interested in a specific disputed fact. Restrict the candidate universe, or specify the maximum number of additions. `stable` is relative to the permitted candidate set; `budget_exceeded` is not proof that a flip is impossible.

In [ ]:
restricted = alice.challenge(TARGET, candidates=["buyer_caused_defect"])
budget = alice.challenge(TARGET, max_changes=1)
assert restricted.status == "stable"
assert budget.status == "budget_exceeded"
print("One allowed candidate:", restricted.status)
print("At most one addition:", budget.status)

## 8. Interactive case lab
In Colab, change the checkboxes below and run the cell again. Treat unchecked propositions as **not established**, not proven false. This is a modeled case assessment, not a real-world decision.

In [ ]:
# @title Set the established facts { run: "auto" }
product_defective = True # @param {type:"boolean"}
purchase_established = True # @param {type:"boolean"}
product_misdescribed = False # @param {type:"boolean"}
buyer_caused_defect = False # @param {type:"boolean"}
damage_after_delivery = False # @param {type:"boolean"}

selected = [name for name, value in zip(FACT_UNIVERSE, [
    product_defective, purchase_established, product_misdescribed,
    buyer_caused_defect, damage_after_delivery]) if value]

def assess_refund(facts):
    case = Reasoner(rules, facts=facts)
    verdict = case.evaluate(TARGET)
    change = case.challenge(TARGET)
    lines = ["### Modeled case assessment",
             "**Entitlement established:** " + str(verdict),
             "| Proposition | Established? |", "|---|---|"]
    for proposition in ["purchase_established", "defect_ground", "product_misdescribed",
                        "damage_exception", "valid_return_ground"]:
        lines.append(f"| `{proposition}` | {case.evaluate(proposition)} |")
    lines += [f"**Counterfactual status:** `{change.status}`"]
    if change.status == "flipped":
        checked = case.verify(change)
        lines += [f"**Minimum additions:** {change.radius}",
                  "**One possible witness:** " + ", ".join(f"`{x}`" for x in change.witness),
                  f"**Valid / optimal:** {checked.valid} / {checked.optimal}"]
    display(Markdown("\n\n".join(lines[:2]) + "\n\n" + "\n".join(lines[2:])))
    return verdict

_ = assess_refund(selected)

Try these combinations: remove purchase evidence; add both exception facts; then establish misdescription as well. Why does the last combination establish entitlement again? A witness identifies hypothetical additions, not evidence that already exists or should be fabricated.

## 9. Test the knowledge base
A test checks whether the implemented interpretation behaves as intended. It does not prove that the interpretation is legally correct. Compare all 32 combinations with a direct Boolean expression for this small example.

In [ ]:
for values in product([False, True], repeat=len(FACT_UNIVERSE)):
    defective, purchase, misdescribed, caused, after = values
    facts = [name for name, value in zip(FACT_UNIVERSE, values) if value]
    expected = purchase and (misdescribed or (defective and not (caused and after)))
    assert evaluate(rules, facts) == expected, facts
print("32 combinations matched the reference expression.")

## 10. Advanced: compare inference methods
The default challenge uses exact cardinality-ordered search. Structural Leap computes minimum plans directly for supported disjoint-leaf structures and falls back to exact search otherwise. Both should return the same minimum for this case; neither is universally faster. An optional Z3 solver is described in the repository documentation.

In [ ]:
leap = alice.analyze(TARGET, method="leap")
assert leap.status == result.status and leap.radius == result.radius
assert alice.verify(leap).optimal is True
print("Exact:", result.radius, result.witness)
print("Leap:", leap.radius, leap.witness)
print("Execution mode:", leap.metadata["execution_mode"])

## 11. Your turn: contract reasoning
A hypothetical contract is voidable if the signer lacked capacity or was coerced. A statutory exception defeats only the incapacity ground. Write a rule graph, then test a case with all three facts established. The answer should still be true because coercion is an independent ground.

**Hint:** use `ANY` for the overall conclusion and attach the exception to the incapacity rule, not the overall conclusion. Uncomment and fill the template before running it.

In [ ]:
# contract_rules = [...]  # TODO: define the three rules
# case = Reasoner(contract_rules, facts=["lacked_capacity", "coerced", "statutory_exception"])
# assert case.evaluate("contract_voidable") is True

## 12. From tutorial to an application
In a real system, maintain separate stages: legal source and version → expert-reviewed rules → evidence validation → PYTHEN evaluation → counterfactual analysis → human review. An LLM may propose candidate facts or rules, but these are not automatically established facts or legally valid rules. Preserve provenance, jurisdiction, dates, disputed evidence, and the limitations of Boolean reasoning.

**Further reading:** [Release v0.2.0a3](https://github.com/nguyenthanhasia/pythen/releases/tag/v0.2.0a3) · [Framework semantics](https://github.com/nguyenthanhasia/pythen/blob/v0.2.0a3/docs/framework_v2.md) · [Source repository](https://github.com/nguyenthanhasia/pythen).